# HDT 1: Pandas, SQL y DuckDB

**Ciencia de Datos, Sección A** · Asignada: martes 28 de julio · **Entrega: martes 4 de agosto, 23:59**

**Nombre:** _(Juan Pablo Madriz)_

Completar las celdas marcadas con `# ¿Qué va aquí?`. Cada ejercicio incluye una verificación comentada: descomentar para comprobar el resultado. Antes de entregar: **Kernel → Restart & Run All** (un notebook que no corre de arriba a abajo pierde 0.5 pts).

AI: resolver sin AI. Si se usó para entender un concepto, anotarlo en la mini-bitácora del final.

## Setup

Si falta DuckDB: descomentar la línea de instalación, ejecutar la celda una vez y volver a comentarla.

In [64]:
# !uv add duckdb    (en terminal)  o descomentar:  %pip install duckdb
import pandas as pd
import duckdb

URL = ("https://raw.githubusercontent.com/"
       "mwaskom/seaborn-data/master/penguins.csv")
penguins = pd.read_csv(URL)

# Tabla de nombres científicos (para los JOIN)
especies = pd.DataFrame({
    "species": ["Adelie", "Chinstrap", "Gentoo"],
    "nombre_cientifico": ["Pygoscelis adeliae",
                          "Pygoscelis antarcticus",
                          "Pygoscelis papua"],
})
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


## Parte A · Pandas (1.0 pt)

### Ejercicio 1 (0.10): cargar y explorar

Mostrar: (a) el `shape` del DataFrame, (b) los `dtypes`, y (c) cuántos nulos tiene **cada columna**.

In [65]:
# ¿Qué va aquí? (tres expresiones, una por inciso)

print(penguins.shape)

# Verificación: el dataset original tiene 344 filas y 7 columnas,

print(penguins.dtypes)

# y la columna sex es la que más nulos tiene (11).
print(f"{penguins.sex.isna().sum()} nulos") # no se usa count() porque cuenta los que tiene valor, y con sum() cuenta los que si tienen Na

(344, 7)
species                  str
island                   str
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                      str
dtype: object
11 nulos


### Ejercicio 2 (0.15): limpieza mínima

Crear un DataFrame `limpio` **sin** las filas que tengan nulo en cualquier columna. Reportar con un `print` cuántas filas se perdieron respecto al original.

In [66]:
limpio = penguins.dropna()
# Verificación (descomentar):
assert limpio.shape[0] == 333 and limpio.isna().sum().sum() == 0
print(f"Se perdieron {penguins.shape[0] - limpio.shape[0]} filas")

Se perdieron 11 filas


### Ejercicio 3 (0.15): máscaras y orden

De `limpio`: los pingüinos de la isla **Biscoe** con masa corporal **mayor a 4500 g**, ordenados de mayor a menor masa. Mostrar solo las columnas `species`, `island`, `body_mass_g`.

In [67]:
pesados_biscoe = limpio[(limpio["island"]=="Biscoe") & (limpio["body_mass_g"] > 4500)] # se uso Ai para investigar porque un and fallaba y realmente se usa &
pesados_biscoe = pesados_biscoe[["species", "island", "body_mass_g"]].sort_values("body_mass_g", ascending=False)
print(pesados_biscoe)

# Verificación (descomentar):
assert (pesados_biscoe["island"] == "Biscoe").all()
assert (pesados_biscoe["body_mass_g"] > 4500).all()
assert pesados_biscoe["body_mass_g"].is_monotonic_decreasing

    species  island  body_mass_g
237  Gentoo  Biscoe       6300.0
253  Gentoo  Biscoe       6050.0
337  Gentoo  Biscoe       6000.0
297  Gentoo  Biscoe       6000.0
331  Gentoo  Biscoe       5950.0
..      ...     ...          ...
306  Gentoo  Biscoe       4600.0
111  Adelie  Biscoe       4600.0
248  Gentoo  Biscoe       4600.0
328  Gentoo  Biscoe       4575.0
225  Gentoo  Biscoe       4550.0

[106 rows x 3 columns]


### Ejercicio 4 (0.20): groupby con dos funciones

Masa corporal por **especie y sexo**: el **promedio** y el **conteo**, en una sola operación con `groupby` + `agg`.

In [68]:
resumen = penguins.groupby(["species", "sex"]).agg(
    promedio = ("body_mass_g", "mean"),
    num_peng = ("species", "count")
)
resumen

# Verificación: el grupo más pesado debe ser Gentoo macho (~5485 g en promedio).

promedio  num_peng
species   sex                          
Adelie    FEMALE  3368.835616        73
          MALE    4043.493151        73
Chinstrap FEMALE  3527.205882        34
          MALE    3938.970588        34
Gentoo    FEMALE  4679.741379        58
          MALE    5484.836066        61

### Ejercicio 5 (0.20): columna derivada

Agregar a `limpio` una columna `bill_ratio` = largo del pico / profundidad del pico. Mostrar el promedio de `bill_ratio` **por especie**, ordenado descendente. ¿Qué especie tiene el pico proporcionalmente más alargado?

In [69]:
# ¿Qué va aquí?
limpio["bill_ratio"] = limpio["bill_length_mm"] / limpio["bill_depth_mm"]
limpio

# Verificación: Gentoo debe quedar de primero (~3.2).

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,bill_ratio
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE,2.090909
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE,2.270115
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE,2.238889
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE,1.901554
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE,1.907767
...,...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,FEMALE,3.445255
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE,3.272727
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE,3.210191
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE,3.054054


### Ejercicio 6 (0.20): merge

Unir `limpio` con la tabla `especies` para que cada fila tenga su `nombre_cientifico`. Mostrar una fila de cada especie para comprobar.

In [70]:
con_nombres = limpio.merge(especies, how="left")
con_nombres

# con_nombres.drop_duplicates("species")[["species", "nombre_cientifico"]]

# Verificación (descomentar):
# assert con_nombres.shape[0] == limpio.shape[0]
# assert "nombre_cientifico" in con_nombres.columns

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,bill_ratio,nombre_cientifico
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE,2.090909,Pygoscelis adeliae
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE,2.270115,Pygoscelis adeliae
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE,2.238889,Pygoscelis adeliae
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE,1.901554,Pygoscelis adeliae
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE,1.907767,Pygoscelis adeliae
...,...,...,...,...,...,...,...,...,...
328,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,FEMALE,3.445255,Pygoscelis papua
329,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE,3.272727,Pygoscelis papua
330,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE,3.210191,Pygoscelis papua
331,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE,3.054054,Pygoscelis papua


## Parte B · SQL con DuckDB (0.8 pt)

DuckDB consulta directamente los DataFrames en memoria: `duckdb.sql("SELECT ... FROM limpio")`. Cerrar cada consulta con `.df()` para ver el resultado como DataFrame.

### Ejercicio 7 (0.20): SELECT / WHERE / ORDER BY

El ejercicio 3, ahora en SQL: especie, isla y masa de los pingüinos de Biscoe con masa mayor a 4500 g, ordenados de mayor a menor.

In [71]:
q7 = """SELECT species, island, body_mass_g
FROM limpio
WHERE island = 'Biscoe' AND body_mass_g > 4500
ORDER BY body_mass_g DESC""" #se uso ai para descubrir un error, y el error fue que tenia doble comillas y es triple

duckdb.sql(q7).df()



,species,island,body_mass_g
0,Gentoo,Biscoe,6300.0
1,Gentoo,Biscoe,6050.0
2,Gentoo,Biscoe,6000.0
3,Gentoo,Biscoe,6000.0
4,Gentoo,Biscoe,5950.0
...,...,...,...
101,Gentoo,Biscoe,4600.0
102,Gentoo,Biscoe,4600.0
103,Adelie,Biscoe,4600.0
104,Gentoo,Biscoe,4575.0


### Ejercicio 8 (0.20): GROUP BY + HAVING

Especies cuya masa corporal **promedio** supera los 4000 g, con su promedio redondeado.

In [72]:
q8 = """
SELECT species, ROUND(AVG(body_mass_g)) as promedio
FROM limpio
GROUP BY species
HAVING AVG(body_mass_g) > 4000

"""
duckdb.sql(q8).df()

# Verificación: solo una especie debe aparecer. ¿Cuál? Comparar con el resultado
# del ejercicio 4.

,species,promedio
0,Gentoo,5092.0


### Ejercicio 9 (0.20): JOIN

El ejercicio 6, ahora en SQL: unir `limpio` con `especies` y mostrar especie, nombre científico y masa promedio por especie.

In [73]:
q9 = """
SELECT limpio.species, especies.nombre_cientifico, AVG(limpio.body_mass_g) as promedio
FROM limpio
JOIN especies ON limpio.species = especies.species
GROUP BY limpio.species, especies.nombre_cientifico
"""
duckdb.sql(q9).df()

# Verificación: 3 filas, una por especie, cada una con su Pygoscelis.

,species,nombre_cientifico,promedio
0,Gentoo,Pygoscelis papua,5092.436975
1,Chinstrap,Pygoscelis antarcticus,3733.088235
2,Adelie,Pygoscelis adeliae,3706.164384


### Ejercicio 10 (0.20): window function

Los **3 pingüinos más pesados de cada especie**, usando `RANK() OVER (PARTITION BY ... ORDER BY ...)`. Pista de la sesión 3: la window function se calcula en una subconsulta y se filtra afuera.

In [74]:
q10 = """
SELECT species, island, body_mass_g, rango
FROM (
    SELECT species, island, body_mass_g,
           RANK() OVER (PARTITION BY species ORDER BY body_mass_g DESC) as rango
    FROM limpio
) sub
WHERE rango <= 3
"""
duckdb.sql(q10).df()

# Verificación: alrededor de 9 filas (3 por especie; puede haber empates),
# y el rango nunca debe pasar de 3.

,species,island,body_mass_g,rango
0,Gentoo,Biscoe,6300.0,1
1,Gentoo,Biscoe,6050.0,2
2,Gentoo,Biscoe,6000.0,3
3,Gentoo,Biscoe,6000.0,3
4,Adelie,Biscoe,4775.0,1
5,Adelie,Biscoe,4725.0,2
6,Adelie,Torgersen,4700.0,3
7,Chinstrap,Dream,4800.0,1
8,Chinstrap,Dream,4550.0,2
9,Chinstrap,Dream,4500.0,3


## Parte C · Criterio (0.2 pt)

### Ejercicio 11 (0.20)

Los mismos análisis se resolvieron en Pandas y en SQL. En 3-4 líneas, **con base en el trabajo de esta hoja** (no de memoria): ¿cuándo conviene cada herramienta? Mencionar al menos una operación que resultó más natural en cada una.

_(Responder editando esta celda)_

**Respuesta:** ...

## Mini-bitácora de AI (opcional, no penaliza)

Si se usó AI para entender algún concepto, anotar aquí qué se preguntó y qué se entendió. Si no se usó, escribir "No se usó".
Se uso AI para el ejercico 10, no entendia la logica ni como usar las funciones que me pedia.
Se uso AI en pandas en los ejercicios que lo dicen. Se uso para preguntar documentación.
- ...

## Anexo: repaso de las sesiones 2 y 3

Regla de los ejercicios de NumPy: **sin ciclos `for`**.

In [75]:
import numpy as np

rng = np.random.default_rng(7)

### A1 (sesión 2): z-score sin loops

Normalizar un array: restar la media y dividir entre la desviación estándar.

In [76]:
alturas = rng.normal(170, 10, size=1000)

def z_score(x):
    # ¿Qué va aquí? (sin for)
    pass

z = z_score(alturas)
# Verificación (descomentar):
# print(round(z.mean(), 4), round(z.std(), 4))  # ~0 y ~1

### A2 (sesión 2): distancias con broadcasting

Distancia euclidiana de cada punto a un centro, sin loops.

In [77]:
puntos = rng.normal(size=(500, 2))   # 500 puntos en 2D
centro = np.array([1.0, 1.0])

# ¿Qué va aquí?
# Pista: (puntos - centro) usa broadcasting (500,2) - (2,)
# Luego: elevar al cuadrado, sumar con axis=1, sacar raíz
distancias = ...

# ¿Cuántos puntos están a menos de 1 del centro?
cercanos = ...

# Verificación (descomentar):
# print(distancias.shape)  # (500,)
# print(cercanos)

### A3 (sesión 3): propinas por día y turno

Dataset `tips` (propinas de un restaurante). `pct` = propina como fracción de la cuenta.

In [78]:
URL_TIPS = ("https://raw.githubusercontent.com/"
            "mwaskom/seaborn-data/master/tips.csv")
tips = pd.read_csv(URL_TIPS)
tips["pct"] = tips["tip"] / tips["total_bill"]
tips.head()

,total_bill,tip,sex,smoker,day,time,size,pct
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059447
1,10.34,1.66,Male,No,Sun,Dinner,3,0.160542
2,21.01,3.50,Male,No,Sun,Dinner,3,0.166587
3,23.68,3.31,Male,No,Sun,Dinner,2,0.139780
4,24.59,3.61,Female,No,Sun,Dinner,4,0.146808


In [79]:
# a) porcentaje medio de propina por dia
por_dia = tips.groupby("day")["pct"].mean()

# b) por dia Y turno (day, time), en una tabla
por_dia_turno = tips.groupby(["day", "time"])["pct"].mean()

# c) el dia con el mayor porcentaje medio
dia_max = por_dia.idxmax()

# d) numero de mesas por dia
resumen = tips.groupby("day").agg(
    pct_promedio=("pct", "mean"),
    num_mesas=("day", "count")
)

print(resumen)
print(f"Día con mayor porcentaje promedio: {dia_max}")
print(por_dia_turno)

# Verificacion: resumen debe tener 4 filas
print(resumen.shape)

      pct_promedio  num_mesas
day                          
Fri       0.169913         19
Sat       0.153152         87
Sun       0.166897         76
Thur      0.161276         62
Día con mayor porcentaje promedio: Fri
day   time  
Fri   Dinner    0.158916
      Lunch     0.188765
Sat   Dinner    0.153152
Sun   Dinner    0.166897
Thur  Dinner    0.159744
      Lunch     0.161301
Name: pct, dtype: float64
(4, 2)
